In [1]:
import os
import sys
from pathlib import Path
import subprocess
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]
sys.path.insert(0, str(Path(os.environ.get("PROJECT_ROOT", "."))))

In [2]:
import glob

training_files = glob.glob(r"D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples/*.xml")
model_path = r"D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/models/segmentation/blla.mlmodel"


In [6]:
from kraken.lib.xml import XMLPage
from kraken import blla

page = XMLPage('D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples/05_garde_001.xml').to_container()
# Then inspect page.lines to see what Kraken parses
print(len(page.lines))

Region eSc_dummyblock_ without coordinates


101


In [3]:
from kraken.lib.xml import XMLPage

for f in training_files:
    page = XMLPage(f).to_container()
    print(f, "→ lines found:", len(page.lines))

Region eSc_dummyblock_ without coordinates
Region eSc_dummyblock_ without coordinates


D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples\05_garde_001.xml → lines found: 101
D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples\17_f_012v_013.xml → lines found: 199


In [12]:
import os
import subprocess

train_file = r"D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples/05_garde_001.xml"
val_file   = r"D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples/17_f_012v_013.xml"

with open("train_manifest.txt", "w") as f:
    f.write(train_file + "\n")
with open("val_manifest.txt", "w") as f:
    f.write(val_file + "\n")

env = os.environ.copy()
env["TERM"] = "dumb"
env["NO_COLOR"] = "1"

try:
    result = subprocess.run(
        [
            "ketos", "segtrain",
            "-f", "xml",
            "-t", "train_manifest.txt",
            "-e", "val_manifest.txt",
            "--load", model_path,
            "--resize", "both",
            "--epochs", "5",
            "--step-size", "1",
            "-o", "models/segmentation/my_occitan_segmentation.mlmodel",
        ],
        check=True,
        capture_output=True,      # capture again so we can print the error
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env
    )
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("STDOUT:\n", e.stdout)
    print("STDERR:\n", e.stderr)  # full traceback will appear here

STDOUT:
 [04/12/26 21:47:32] WARNING  Region eSc_dummyblock_ without           xml.py:321
                             coordinates                                        
                    WARNING  Region eSc_dummyblock_ without           xml.py:321
                             coordinates                                        
Training line types:
  default	2	101
Training region types:
  text	3	1
                    WARNING  Setting baseline location to baseline train.py:1032
                             from unset model.                                  
┌────┬───────────────────┬───────────┬────────┬───────┬────────────┬───────────┐
│    │ Name              │ Type      │ Params │ Mode  │   In sizes │ Out sizes │
├────┼───────────────────┼───────────┼────────┼───────┼────────────┼───────────┤
│ 0  │ net               │ MultiPar… │  1.3 M │ train │     [1, 3, │   [[1, 4, │
│    │                   │           │        │       │ 1800, 300] │ 450, 75], │
│    │                   │   

In [13]:
from kraken.lib.train import SegmentationModel
from kraken.lib.xml import XMLPage

# Quick check — confirm pages are loading correctly
for path in [train_file, val_file]:
    page = XMLPage(path).to_container()
    print(f"{path}: {len(page.lines)} lines")

Region eSc_dummyblock_ without coordinates
Region eSc_dummyblock_ without coordinates


D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples/05_garde_001.xml: 101 lines
D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples/17_f_012v_013.xml: 199 lines


In [2]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]
sys.path.insert(0, str(Path(os.environ.get("PROJECT_ROOT", "."))))

# ── Patch Rich console ───────────────────────────────────────────────────────
import rich.console
_original_clear_live = rich.console.Console.clear_live
def _safe_clear_live(self):
    try:
        _original_clear_live(self)
    except IndexError:
        pass
rich.console.Console.clear_live = _safe_clear_live
# ────────────────────────────────────────────────────────────────────────────

import lightning.pytorch as pl
from kraken.lib.train import SegmentationModel

train_file = r"D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples/05_garde_001.xml"
val_file   = r"D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/annotated_samples/17_f_012v_013.xml"
model_path = r"D:/Users/kju10/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/models/segmentation/blla.mlmodel"
output     = "models/segmentation/my_occitan_segmentation"

hyper_params = {
    'line_width': 8,
    'padding': (0, 0),
    'freq': 1.0,
    'quit': 'fixed',
    'epochs': 5,
    'min_epochs': 0,
    'lag': 10,
    'min_delta': None,
    'optimizer': 'Adam',
    'lrate': 2e-4,
    'momentum': 0.9,
    'weight_decay': 1e-5,
    'warmup': 0,
    'schedule': 'constant',
    'gamma': 0.1,
    'step_size': 1,
    'sched_patience': 5,
    'cos_max': 50,
    'cos_min_lr': 2e-5,
    'batch_size': 1,
    'augment': False,
}

model = SegmentationModel(
    hyper_params,
    output=output,
    model=model_path,
    training_data=[train_file],
    evaluation_data=[val_file],
    format_type='xml',
    partition=0.9,
    num_workers=0,          # was 0, increase for faster data loading
    force_binarization=False,
    suppress_regions=False,
    suppress_baselines=False,
    valid_regions=None,
    valid_baselines=None,
    merge_regions=None,
    merge_baselines=None,
    bounding_regions=None,
    resize='both',
    topline=False,
)

trainer = pl.Trainer(
    max_epochs=5,
    enable_progress_bar=False,
    logger=False,
    callbacks=[],           # suppresses the pickle warnings
)

trainer.fit(model)
print("Training complete!")

Region eSc_dummyblock_ without coordinates
Region eSc_dummyblock_ without coordinates
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
d:\Users\kju10\Documents\LMU-STATISTICS & DATA SCIENCE MASTER\SS2026\Thesis\OCC_HTR\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:44: Attribute 'progress_callback' removed from hparams because it cannot be pickled. You can suppress this warning by setting `self.save_hyperparameters(ignore=['progress_callback'])`.
d:\Users\kju10\Documents\LMU-STATISTICS & DATA SCIENCE MASTER\SS2026\Thesis\OCC_HTR\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:44: Attribute 'message' removed from hparams because it cannot be pickled. You can suppress this warning by setting `self.save_hyperparameters(ignore=['message'])`.
Setting baseline location to baseline from unset model.

  | Name              | Type                   | Params | Mode  | In sizes          | Out sizes       

: 

In [7]:
import subprocess
result = subprocess.run(["ketos", "segtrain", "--help"], capture_output=True, text=True)
print(result.stdout)

Usage: ketos segtrain [OPTIONS] [GROUND_TRUTH]...

  Trains a baseline labeling model for layout analysis

Options:
  -o, --output PATH               Output model file  [default: model]
  -s, --spec TEXT                 VGSL spec of the baseline labeling network
                                  [default: [1,1800,0,3 Cr7,7,64,2,2 Gn32
                                  Cr3,3,128,2,2 Gn32 Cr3,3,128 Gn32 Cr3,3,256
                                  Gn32 Cr3,3,256 Gn32 Lbx32 Lby32 Cr1,1,32
                                  Gn32 Lby32 Lbx32]]
  --line-width INTEGER            The height of each baseline in the target
                                  after scaling  [default: 8]
  --pad <INTEGER INTEGER>...      Padding (left/right, top/bottom) around the
                                  page image  [default: 0, 0]
  -i, --load PATH                 Load existing file to continue training
  -F, --freq FLOAT                Model saving and report generation frequency
                          